# Weekend 2 — Python: pliki, wyjątki, mini-CLI

## Ćwiczenia solved

Ten notebook zawiera kompletne rozwiązania krok po kroku do zadań z obsługi plików tekstowych, wyjątków (`FileNotFoundError`) oraz prostego interfejsu **mini-CLI** opartego na `sys.argv`.

In [ ]:
from pathlib import Path
import sys
import subprocess

NOTES_DIR = Path("notes_box")
NOTES_DIR.mkdir(exist_ok=True)
SAMPLE_NOTE = NOTES_DIR / "dziennik.txt"
SAMPLE_NOTE.write_text("1. Uczyć się podstaw Pythona.\n2. Zrozumieć wyjątki i try-except.\n3. Napisać pierwsze mini-CLI.\n", encoding="utf-8")

print("Gotowy katalog testowy:", NOTES_DIR.exists())

## Rozwiązanie Zadania 1 — Odczyt pliku (`read_note_file`)

In [ ]:
def read_note_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

print("Zawartość notatki:\n" + read_note_file(SAMPLE_NOTE))

## Rozwiązanie Zadania 2 — Bezpieczny odczyt z obsługą wyjątków (`safe_read_note`)

In [ ]:
def safe_read_note(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Błąd: Plik '{file_path}' nie został znaleziony."

print(safe_read_note(SAMPLE_NOTE))
print(safe_read_note("brak_pliku.txt"))

## Rozwiązanie Zadania 3 — Dopisywanie notatki (`append_note`)

In [ ]:
def append_note(file_path, new_line):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(new_line + "\n")

append_note(SAMPLE_NOTE, "4. Przetestować mini-CLI.")
print(read_note_file(SAMPLE_NOTE))

## Rozwiązanie Zadania 4 — Obsługa komend mini-CLI (`handle_command`)

In [ ]:
def handle_command(args):
    if len(args) < 2:
        return "Błąd: Brak wymaganych argumentów."
    
    action = args[0]
    filepath = args[1]
    
    try:
        content = read_note_file(filepath)
    except FileNotFoundError:
        return f"Błąd: Plik '{filepath}' nie istnieje."
        
    if action == "read":
        return content
    elif action == "count":
        lines = content.splitlines()
        return f"Liczba linijek w pliku: {len(lines)}"
    else:
        return f"Nieznana akcja: {action}"

print(handle_command(["read", str(SAMPLE_NOTE)]))
print(handle_command(["count", str(SAMPLE_NOTE)]))

## Rozwiązania Zadań 5 i 6 — Skrypt `notes_cli.py` i testy w subprocess

In [ ]:
CLI_SCRIPT = Path("notes_cli.py")
cli_code = '''
import sys

def main():
    if len(sys.argv) < 3:
        print("Użycie: python notes_cli.py [read|count] <plik>")
        sys.exit(1)
        
    action = sys.argv[1]
    filepath = sys.argv[2]
    
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
    except FileNotFoundError:
        print(f"Błąd: Plik '{filepath}' nie istnieje.")
        sys.exit(2)
        
    if action == "read":
        print(content)
    elif action == "count":
        lines = content.splitlines()
        print(f"Liczba linijek w pliku: {len(lines)}")
    else:
        print(f"Nieznana akcja: {action}")
        sys.exit(1)

if __name__ == "__main__":
    main()
'''
CLI_SCRIPT.write_text(cli_code.strip() + "\n", encoding="utf-8")

res1 = subprocess.run([sys.executable, "notes_cli.py", "read", str(SAMPLE_NOTE)], capture_output=True, text=True)
print("STDOUT (read):\n", res1.stdout)
res3 = subprocess.run([sys.executable, "notes_cli.py", "read", "brak.txt"], capture_output=True, text=True)
print("STDERR / STDOUT (brak pliku):\n", res3.stdout or res3.stderr)
print("Kod wyjścia dla brakującego pliku:", res3.returncode)

# Podsumowanie rozwiązań
Masz teraz w pełni działające, proste mini-CLI oparte na plikach i wyjątkach – dokładnie w zakresie wymaganym na Weekend 1 (Blok 4).